# Check of the Photometric Calibration Law $F_{nJy} = C_{local} \cdot F_{instrumental}$

This notebook checks, for the enriched light-curve table produced by
`05_AddLSSTCalibParam.ipynb`, that the calibrated flux in nanoJansky is
correctly recovered from the instrumental flux and the local calibration
factor:

$$
F_{nJy} \;=\; C_{\mathrm{local}}(x,y,b) \cdot F_{\mathrm{instrumental}}
$$

where `calib_local` is the local photometric calibration factor (band- and
position-dependent) attached to each source.

## Flux types checked

Three aperture-flux pairs have both a calibrated (nJy) column and an
instrumental-flux column in the input table, and are checked with the full
bissectrice + residuals diagnostic:

| Label | Calibrated (nJy) | Instrumental |
|---|---|---|
| ap12 | `ap12Flux` / `ap12FluxErr` | `apFlux_12_0_instFlux` / `apFlux_12_0_instFluxErr` |
| ap17 | `ap17Flux` / `ap17FluxErr` | `apFlux_17_0_instFlux` / `apFlux_17_0_instFluxErr` |
| ap35 | `ap35Flux` / `ap35FluxErr` | `apFlux_35_0_instFlux` / `apFlux_35_0_instFluxErr` |

`localBackground_instFlux` has **no calibrated (nJy) counterpart** in the
table, so the bissectrice/residuals check cannot be applied to it; instead
a simple diagnostic of `calib_local * localBackground_instFlux` is shown
(see last section).

## What this notebook shows

For **each flux type** (ap12, ap17, ap35), two figures are produced, each
with **6 subplots** (2 rows x 3 columns, one per band u, g, r, i, z, y):

1. **Bissectrice check**: scatter of $F_{nJy}$ (Y) vs. $C_{\mathrm{local}}
   \cdot F_{\mathrm{instrumental}}$ (X), points coloured by `detector_id`,
   with the $y=x$ line overlaid. Detectors that visibly depart from the
   bissectrice indicate a calibration problem for that CCD.
2. **Residuals check**: scatter + errorbar of
   $F_{nJy} - C_{\mathrm{local}} \cdot F_{\mathrm{instrumental}}$ (Y) vs.
   $C_{\mathrm{local}} \cdot F_{\mathrm{instrumental}}$ (X), with X and Y
   error bars, points coloured by `detector_id`.

## Input
- `data_AddCalib_05_out/all_stars_lightcurves_calib.csv`

## Output
- `figs_CheckCalibCoeff_07/calibcheck_scatter_<flux_label>_all_bands.pdf/.png`
- `figs_CheckCalibCoeff_07/calibcheck_residuals_<flux_label>_all_bands.pdf/.png`
- `figs_CheckCalibCoeff_07/calibcheck_localbackground_all_bands.pdf/.png`

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-30
- **Last update:** 2026-06-30


## 1. Imports

In [ ]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → %matplotlib inline")

## 2. Logging setup

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ─────────────────────────────────────────────────────────────
NB_TAG = "CheckCalibCoeff_07"

# ── Input: enriched light curves with calibration (output of notebook 05) ────
DIR_DATA_IN = "./data_AddCalib_05_out"
GLOBAL_CALIB_FILE = "all_stars_lightcurves_calib.csv"

# ── Output ────────────────────────────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Directory ready: %s", DIR_FIGS)

# ── Band order (LSST community convention) ──────────────────────────────────
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]

# ── Aperture-flux pairs checked with the full bissectrice + residuals diagnostic
# Each entry: label, calibrated (nJy) flux/err columns, instrumental flux/err columns
FLUX_TYPES = [
    dict(
        label="ap12",
        nJy="ap12Flux",
        nJyErr="ap12FluxErr",
        inst="apFlux_12_0_instFlux",
        instErr="apFlux_12_0_instFluxErr",
    ),
    dict(
        label="ap17",
        nJy="ap17Flux",
        nJyErr="ap17FluxErr",
        inst="apFlux_17_0_instFlux",
        instErr="apFlux_17_0_instFluxErr",
    ),
    dict(
        label="ap35",
        nJy="ap35Flux",
        nJyErr="ap35FluxErr",
        inst="apFlux_35_0_instFlux",
        instErr="apFlux_35_0_instFluxErr",
    ),
]

# ── localBackground: instrumental flux only, no calibrated (nJy) counterpart ─
LOCALBACKGROUND_INST = "localBackground_instFlux"
LOCALBACKGROUND_INST_ERR = "localBackground_instFluxErr"

log.info("Configuration done. %d flux types to check: %s", len(FLUX_TYPES), [f["label"] for f in FLUX_TYPES])

## 4. Load the enriched light-curve table

In [ ]:
data_path = os.path.join(DIR_DATA_IN, GLOBAL_CALIB_FILE)
log.info("Loading: %s", data_path)
df = pd.read_csv(data_path)
log.info("Shape: %s", df.shape)
df.head(3)

In [ ]:
# Verify required columns are present
REQUIRED_COLS = ["band", "calib_local", "detector_id", "detector_name"]
for ft in FLUX_TYPES:
    REQUIRED_COLS += [ft["nJy"], ft["nJyErr"], ft["inst"], ft["instErr"]]
REQUIRED_COLS += [LOCALBACKGROUND_INST, LOCALBACKGROUND_INST_ERR]

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing from input file: {missing}")
log.info("All required columns present (%d columns checked).", len(REQUIRED_COLS))

# Keep only rows with a valid local calibration and detector id
n_before = len(df)
df = df.dropna(subset=["calib_local", "detector_id", "band"]).copy()
df["detector_id"] = df["detector_id"].astype(int)
n_after = len(df)
log.info(
    "Rows with valid calib_local/detector_id: %d / %d (%.1f %%)",
    n_after,
    n_before,
    100.0 * n_after / n_before if n_before else 0,
)

## 5. Bands and detectors present in the data

In [ ]:
bands_available = [b for b in BAND_ORDER if b in df["band"].unique()]
extra_bands = sorted(set(df["band"].unique()) - set(BAND_ORDER))
if extra_bands:
    log.warning("Unexpected bands found (not in standard ugrizy list): %s", extra_bands)
    bands_available += extra_bands

log.info("Bands to plot (%d): %s", len(bands_available), bands_available)

# Shared detector_id normalisation for the colour scale, common to ALL figures
DET_ID_MIN = df["detector_id"].min()
DET_ID_MAX = df["detector_id"].max()
DET_NORM = matplotlib.colors.Normalize(vmin=DET_ID_MIN, vmax=DET_ID_MAX)
DET_CMAP = plt.get_cmap("nipy_spectral")

log.info(
    "Detector id range: [%d, %d] (%d distinct detectors)", DET_ID_MIN, DET_ID_MAX, df["detector_id"].nunique()
)
df["band"].value_counts().reindex(bands_available)

## 6. Figure 1 — Bissectrice check: $F_{nJy}$ vs. $C_{local} \cdot F_{instrumental}$

One figure per flux type, 6 subplots (2 rows x 3 cols, bands u, g, r, i, z,
y). Points are coloured by `detector_id` (shared colour scale across all
figures in this notebook). The black dashed line is the $y=x$ bissectrice:
points/detectors that depart from it indicate a calibration problem for
those detectors.

In [ ]:
def plot_calibcheck_scatter_all_bands(
    df: pd.DataFrame,
    bands: list,
    flux_type: dict,
    dir_figs: str,
) -> pd.DataFrame:
    """Bissectrice-check figure for one flux type: 6 subplots (2x3), one per
    band. X = calib_local * F_instrumental, Y = F_nJy, colour = detector_id.
    Returns a per-band/per-detector outlier summary (median relative
    deviation from the bissectrice).
    """
    label = flux_type["label"]
    nJy_col, inst_col = flux_type["nJy"], flux_type["inst"]

    dfc = df.dropna(subset=[nJy_col, inst_col, "calib_local", "detector_id"]).copy()
    dfc["F_pred"] = dfc["calib_local"] * dfc[inst_col]

    bands_toplot = [b for b in bands if (dfc["band"] == b).any()]
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 9))
    axes_flat = axes.flatten()

    outlier_rows = []
    sc = None
    for ax, band in zip(axes_flat, bands_toplot):
        sub = dfc.loc[dfc["band"] == band]
        x = sub["F_pred"].to_numpy()
        y = sub[nJy_col].to_numpy()
        det = sub["detector_id"].to_numpy()

        sc = ax.scatter(x, y, c=det, cmap=DET_CMAP, norm=DET_NORM, s=10, alpha=0.7, edgecolors="none")

        if len(x) > 0:
            lo = min(x.min(), y.min())
            hi = max(x.max(), y.max())
            ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x")
            ax.set_xlim(lo, hi)
            ax.set_ylim(lo, hi)

        ax.set_xlabel(r"$C_{local} \cdot F_{instrumental}$  [nJy]", fontsize=9)
        ax.set_ylabel(rf"${nJy_col}$  [nJy]", fontsize=9)
        ax.set_title(f"band '{band}'  ({len(sub)} pts)", fontsize=11)
        ax.legend(loc="upper left", fontsize=7)
        ax.grid(True, alpha=0.3)

        # per-detector median relative deviation from the bissectrice
        sub2 = sub.copy()
        sub2["rel_dev"] = (sub2[nJy_col] - sub2["F_pred"]) / sub2["F_pred"]
        per_det = sub2.groupby("detector_id")["rel_dev"].agg(n="count", median="median", std="std")
        per_det["band"] = band
        per_det["flux_type"] = label
        outlier_rows.append(per_det.reset_index())

    for ax in axes_flat[len(bands_toplot) :]:
        ax.set_visible(False)

    fig.suptitle(
        rf"Bissectrice check ($F_{{nJy}}$ vs $C_{{local}}\cdot F_{{instrumental}}$) — "
        rf"flux type '{label}'  (point colour = detector_id)",
        fontsize=14,
        y=0.995,
    )
    # Reserve room on the right for a dedicated colorbar axis, then place it
    # there explicitly (more robust than ax=list(axes_flat), which can end up
    # centred once unused subplots are hidden).
    plt.tight_layout(rect=[0, 0, 0.91, 0.96])
    if sc is not None:
        cax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
        cbar = fig.colorbar(sc, cax=cax)
        cbar.set_label("detector_id")

    out_name = f"calibcheck_scatter_{label}_all_bands"
    fig.savefig(os.path.join(dir_figs, out_name + ".pdf"), dpi=150, bbox_inches="tight")
    fig.savefig(os.path.join(dir_figs, out_name + ".png"), dpi=150, bbox_inches="tight")
    log.info("Figure saved: %s (.pdf/.png)", os.path.join(dir_figs, out_name))
    plt.show()

    return pd.concat(outlier_rows, ignore_index=True) if outlier_rows else pd.DataFrame()

In [ ]:
outlier_summaries = []
for ft in FLUX_TYPES:
    log.info("Plotting bissectrice check for flux type '%s'...", ft["label"])
    outlier_summaries.append(plot_calibcheck_scatter_all_bands(df, bands_available, ft, DIR_FIGS))

df_outliers = pd.concat(outlier_summaries, ignore_index=True)
log.info("Outlier summary table built: %d (flux_type, band, detector) rows.", len(df_outliers))

## 7. Figure 2 — Residuals: $F_{nJy} - C_{local}\cdot F_{instrumental}$ vs. $C_{local}\cdot F_{instrumental}$

One figure per flux type, 6 subplots (2 rows x 3 cols, bands u, g, r, i, z,
y). Error bars (light gray) propagate both the nJy-flux error and the
instrumental-flux error (scaled by `calib_local`); points on top are
coloured by `detector_id` (same colour scale as Figure 1).

In [ ]:
def plot_calibcheck_residuals_all_bands(
    df: pd.DataFrame,
    bands: list,
    flux_type: dict,
    dir_figs: str,
) -> None:
    """Residuals figure for one flux type: 6 subplots (2x3), one per band.
    X = calib_local * F_instrumental, Y = F_nJy - X, with X/Y error bars.
    Points coloured by detector_id (errorbars drawn underneath, light gray).
    """
    label = flux_type["label"]
    nJy_col, nJyErr_col = flux_type["nJy"], flux_type["nJyErr"]
    inst_col, instErr_col = flux_type["inst"], flux_type["instErr"]

    dfc = df.dropna(subset=[nJy_col, nJyErr_col, inst_col, instErr_col, "calib_local", "detector_id"]).copy()
    dfc["F_pred"] = dfc["calib_local"] * dfc[inst_col]
    dfc["F_pred_err"] = dfc["calib_local"] * dfc[instErr_col]
    dfc["residual"] = dfc[nJy_col] - dfc["F_pred"]
    dfc["residual_err"] = np.sqrt(dfc[nJyErr_col] ** 2 + dfc["F_pred_err"] ** 2)

    bands_toplot = [b for b in bands if (dfc["band"] == b).any()]
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 9))
    axes_flat = axes.flatten()

    sc = None
    for ax, band in zip(axes_flat, bands_toplot):
        sub = dfc.loc[dfc["band"] == band]
        x = sub["F_pred"].to_numpy()
        y = sub["residual"].to_numpy()
        xerr = sub["F_pred_err"].to_numpy()
        yerr = sub["residual_err"].to_numpy()
        det = sub["detector_id"].to_numpy()

        ax.errorbar(
            x, y, xerr=xerr, yerr=yerr, fmt="none", ecolor="lightgray", elinewidth=0.5, alpha=0.6, zorder=1
        )
        sc = ax.scatter(
            x, y, c=det, cmap=DET_CMAP, norm=DET_NORM, s=10, alpha=0.8, edgecolors="none", zorder=2
        )
        ax.axhline(0.0, color="black", lw=1, ls="--")

        ax.set_xlabel(r"$C_{local} \cdot F_{instrumental}$  [nJy]", fontsize=9)
        ax.set_ylabel(rf"${nJy_col} - C_{{local}}\cdot F_{{instrumental}}$  [nJy]", fontsize=8)
        ax.set_title(f"band '{band}'  ({len(sub)} pts)", fontsize=11)
        ax.grid(True, alpha=0.3)

    for ax in axes_flat[len(bands_toplot) :]:
        ax.set_visible(False)

    fig.suptitle(
        rf"Residuals ($F_{{nJy}} - C_{{local}}\cdot F_{{instrumental}}$) — "
        rf"flux type '{label}'  (point colour = detector_id)",
        fontsize=14,
        y=0.995,
    )
    plt.tight_layout(rect=[0, 0, 0.91, 0.96])
    if sc is not None:
        cax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
        cbar = fig.colorbar(sc, cax=cax)
        cbar.set_label("detector_id")

    out_name = f"calibcheck_residuals_{label}_all_bands"
    fig.savefig(os.path.join(dir_figs, out_name + ".pdf"), dpi=150, bbox_inches="tight")
    fig.savefig(os.path.join(dir_figs, out_name + ".png"), dpi=150, bbox_inches="tight")
    log.info("Figure saved: %s (.pdf/.png)", os.path.join(dir_figs, out_name))
    plt.show()

In [ ]:
for ft in FLUX_TYPES:
    log.info("Plotting residuals check for flux type '%s'...", ft["label"])
    plot_calibcheck_residuals_all_bands(df, bands_available, ft, DIR_FIGS)

## 8. Detectors without a bissectrice — numerical flag

For each (flux type, band, detector), the **median relative deviation**
$\mathrm{median}\!\left[(F_{nJy} - F_{pred})/F_{pred}\right]$ is computed
(from Figure 1's underlying data). Detectors flagged below have a median
relative deviation larger than `OUTLIER_THRESHOLD` (in absolute value) and
at least `MIN_POINTS` points — i.e. detectors for which the data
systematically departs from the $y=x$ bissectrice, suggesting a local
calibration issue rather than mere noise.

In [ ]:
OUTLIER_THRESHOLD = 0.01  # 1% systematic relative deviation
MIN_POINTS = 5

flagged = df_outliers.loc[
    (df_outliers["n"] >= MIN_POINTS) & (df_outliers["median"].abs() > OUTLIER_THRESHOLD)
].sort_values("median", key=lambda s: s.abs(), ascending=False)

log.info(
    "Flagged %d / %d (flux_type, band, detector) combinations with |median rel. dev.| > %.1f %% (n >= %d).",
    len(flagged),
    len(df_outliers),
    100 * OUTLIER_THRESHOLD,
    MIN_POINTS,
)
display(flagged)

csv_path = os.path.join(DIR_FIGS, "calibcheck_flagged_detectors.csv")
df_outliers.sort_values(["flux_type", "band", "detector_id"]).to_csv(csv_path, index=False)
log.info("Full per-detector deviation table saved: %s", csv_path)

## 9. `localBackground` — no calibrated (nJy) reference column

There is no `localBackgroundFlux` column in nJy in the input table, so the
bissectrice/residuals check used above for the aperture fluxes cannot be
applied here. As a diagnostic, the calibrated local background
$C_{local} \cdot F_{localBackground,\,instrumental}$ is shown vs. MJD,
points coloured by `detector_id` (same colour scale as the figures above).

In [ ]:
def plot_localbackground_diagnostic_all_bands(
    df: pd.DataFrame,
    bands: list,
    dir_figs: str,
) -> None:
    """6 subplots (2x3), one per band: calib_local * localBackground_instFlux
    vs MJD, coloured by detector_id. No nJy ground truth is available for
    this quantity, so this is a distribution check only (no bissectrice).
    """
    dfc = df.dropna(subset=[LOCALBACKGROUND_INST, "calib_local", "detector_id", "expMidptMJD"]).copy()
    dfc["localBackground_calib"] = dfc["calib_local"] * dfc[LOCALBACKGROUND_INST]

    bands_toplot = [b for b in bands if (dfc["band"] == b).any()]
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 9))
    axes_flat = axes.flatten()

    sc = None
    for ax, band in zip(axes_flat, bands_toplot):
        sub = dfc.loc[dfc["band"] == band].sort_values("expMidptMJD")
        sc = ax.scatter(
            sub["expMidptMJD"],
            sub["localBackground_calib"],
            c=sub["detector_id"],
            cmap=DET_CMAP,
            norm=DET_NORM,
            s=10,
            alpha=0.7,
            edgecolors="none",
        )
        ax.axhline(0.0, color="black", lw=1, ls="--")
        ax.set_xlabel("MJD", fontsize=9)
        ax.set_ylabel(r"$C_{local} \cdot F_{localBackground,\,instrumental}$  [nJy]", fontsize=8)
        ax.set_title(f"band '{band}'  ({len(sub)} pts)", fontsize=11)
        ax.grid(True, alpha=0.3)

    for ax in axes_flat[len(bands_toplot) :]:
        ax.set_visible(False)

    fig.suptitle(
        "Calibrated local background vs. MJD — all bands  (no nJy reference available; "
        "point colour = detector_id)",
        fontsize=13,
        y=0.995,
    )
    plt.tight_layout(rect=[0, 0, 0.91, 0.96])
    if sc is not None:
        cax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
        cbar = fig.colorbar(sc, cax=cax)
        cbar.set_label("detector_id")

    out_name = "calibcheck_localbackground_all_bands"
    fig.savefig(os.path.join(dir_figs, out_name + ".pdf"), dpi=150, bbox_inches="tight")
    fig.savefig(os.path.join(dir_figs, out_name + ".png"), dpi=150, bbox_inches="tight")
    log.info("Figure saved: %s (.pdf/.png)", os.path.join(dir_figs, out_name))
    plt.show()


plot_localbackground_diagnostic_all_bands(df, bands_available, DIR_FIGS)

## 10. Output directory listing

In [ ]:
print(f"\n=== Contents of {DIR_FIGS} ===")
for entry in sorted(os.listdir(DIR_FIGS)):
    full = os.path.join(DIR_FIGS, entry)
    size_kb = os.path.getsize(full) / 1024
    print(f"  [FILE] {entry}  ({size_kb:.1f} kB)")